In [ ]:
import requests
import json
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

def get_and_save_results(url, cookie_env_var="METRO_CUADRADO_COOKIE", api_key_env_var="METRO_CUADRADO_X_API_KEY", output_file="results.json"):
    cookie = os.getenv(cookie_env_var)
    api_key = os.getenv(api_key_env_var)
    
    if not cookie:
        raise ValueError(f"Cookie not found in environment variable '{cookie_env_var}'. Please set it in your .env file.")
    
    if not api_key:
        raise ValueError(f"API key not found in environment variable '{api_key_env_var}'. Please set it in your .env file.")
    
    headers = {
        "Cookie": cookie,
        "x-api-key": api_key
    }
    
    print(f"Making GET request to: {url}")
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    data = response.json()
    
    if "results" not in data:
        raise KeyError("'results' key not found in response")
    
    results = data["results"]
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"Saved {len(results)} results to {output_file}")
    return results

In [ ]:
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

def get_all_results_paginated(base_url, limit, page_size=50, cookie_env_var="METRO_CUADRADO_COOKIE", api_key_env_var="METRO_CUADRADO_X_API_KEY", output_folder="./sources/", current_from=0):
    parsed_url = urlparse(base_url)
    query_params = parse_qs(parsed_url.query)
    
    all_results = []
    current_from = current_from
    
    os.makedirs(output_folder, exist_ok=True)
    
    while current_from < limit:
        query_params['from'] = [str(current_from)]
        query_params['size'] = [str(page_size)]
        
        new_query = urlencode(query_params, doseq=True)
        updated_url = urlunparse((
            parsed_url.scheme,
            parsed_url.netloc,
            parsed_url.path,
            parsed_url.params,
            new_query,
            parsed_url.fragment
        ))
        
        output_file = os.path.join(output_folder, f"metro_cuadrado_{current_from}.json")
        
        print(f"\nFetching page starting at offset {current_from}...")
        results = get_and_save_results(
            updated_url,
            cookie_env_var=cookie_env_var,
            api_key_env_var=api_key_env_var,
            output_file=output_file
        )
        
        all_results.extend(results)
        
        if len(results) < page_size:
            print(f"Received {len(results)} results (less than page size). Reached end of data.")
            break
        
        current_from += page_size
    
    combined_output = os.path.join(output_folder, "metro_cuadrado_all.json")
    with open(combined_output, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    
    print(f"\n=== Summary ===")
    print(f"Total pages fetched: {len(all_results) // page_size + (1 if len(all_results) % page_size else 0)}")
    print(f"Total results: {len(all_results)}")
    print(f"Combined results saved to: {combined_output}")
    
    return all_results

In [ ]:
url = "https://www.metrocuadrado.com/rest-search/search?size=50&from=0&realEstateBusinessList=arriendo&city=bogota"
results = get_and_save_results(url,  cookie_env_var="METRO_CUADRADO_COOKIE", api_key_env_var="METRO_CUADRADO_X_API_KEY", output_file="./sources/metro_cuadrado0.json")

In [16]:
base_url = "https://www.metrocuadrado.com/rest-search/search?size=500&from=0&realEstateBusinessList=arriendo&city=bogota"
all_results = get_all_results_paginated(base_url, limit=33950, page_size=50)    


Fetching page starting at offset 10000...
Making GET request to: https://www.metrocuadrado.com/rest-search/search?size=50&from=10000&realEstateBusinessList=arriendo&city=bogota
Saved 50 results to ./sources/metro_cuadrado_10000.json

Fetching page starting at offset 10050...
Making GET request to: https://www.metrocuadrado.com/rest-search/search?size=50&from=10050&realEstateBusinessList=arriendo&city=bogota


HTTPError: 500 Server Error: Internal Server Error for url: https://www.metrocuadrado.com/rest-search/search?size=50&from=10050&realEstateBusinessList=arriendo&city=bogota